# Milestone 2: RAG Exploration Notebook

This notebook documents the exploratory work for **Milestone 2: RAG Pipeline — Semantic and Hybrid Retrieval** using the **All Beauty** category and the cleaned retrieval dataset produced in Milestone 1.

## Notebook goals
- load and validate the cleaned All Beauty dataset
- justify the document representation used for retrieval and RAG
- reuse the **Milestone 1 query set** for continuity
- instantiate **BM25**, **semantic**, and **hybrid** retrievers
- compare retrieval behavior on representative queries
- run **semantic RAG** and **hybrid RAG**
- compare **prompt variants**
- prepare a qualitative evaluation template for `results/milestone2_discussion.md`

## Why this notebook is structured this way
The milestone brief asks for:
1. a text-generation pipeline,
2. semantic retrieval for RAG,
3. a hybrid retriever built from BM25 + semantic search,
4. prompt experimentation,
5. initial explorations in `notebooks/milestone2_rag.ipynb`, and
6. manual qualitative evaluation on queries created in Milestone 1.

This notebook is therefore designed to mirror that workflow from data loading all the way to answer generation and evaluation.

## 1. Imports and setup

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from src.bm25 import BM25Retriever
from src.hybrid import HybridRetriever
from src.rag_pipeline import (
    LLMPipeline,
    PROMPT_VARIANTS,
    build_context,
    build_rag_chain,
    build_semantic_vectorstore,
)
from src.semantic import SemanticRetriever
from src.preprocessing import find_repo_root

C:\Users\ruthy\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

pd.set_option("display.max_colwidth", 140)

## 2. Locate the repository root and load the cleaned dataset

In [3]:
def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root by walking upward until key project folders exist.

    Parameters
    ----------
    start : pathlib.Path or None, default=None
        Starting path for the upward search. If None, the current working
        directory is used.

    Returns
    -------
    pathlib.Path
        Repository root path.

    Raises
    ------
    FileNotFoundError
        If no suitable repository root can be found.
    """
    current = (start or Path.cwd()).resolve()

    for path in [current, *current.parents]:
        if (path / "data").exists() and (path / "src").exists():
            return path

    raise FileNotFoundError(
        "Could not find repository root containing both `data/` and `src/`."
    )


PROJECT_ROOT = find_repo_root()
DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR =", DATA_DIR)
print("RESULTS_DIR =", RESULTS_DIR)

PROJECT_ROOT = C:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray
DATA_DIR = C:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\data
RESULTS_DIR = C:\Users\ruthy\assignments\block6\group_575\DSCI_575_project_omo001_deepray\results


In [4]:
DATA_PATH = DATA_DIR / "processed" / "All_Beauty_clean.parquet"
DATA_PATH

WindowsPath('C:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray/data/processed/All_Beauty_clean.parquet')

In [5]:
def load_reviews(path: Path) -> pd.DataFrame:
    """Load the cleaned retrieval dataset.

    Parameters
    ----------
    path : pathlib.Path
        Path to the dataset file.

    Returns
    -------
    pandas.DataFrame
        Loaded review dataframe.

    Raises
    ------
    FileNotFoundError
        If the file does not exist.
    ValueError
        If the file format is unsupported.
    """
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    if suffix in {".jsonl", ".json"}:
        return pd.read_json(path, lines=True)

    raise ValueError(
        "Unsupported dataset format. Please provide a CSV, parquet, or JSONL file."
    )


df = load_reviews(DATA_PATH)
df.head()

,doc_id,parent_asin,asin,title,rating,text
0,B00YQ6X8EO_0,B00YQ6X8EO,B00YQ6X8EO,Such a lovely scent but not overpowering.,5,"such a lovely scent but not overpowering. this spray is really nice. it smells really good, goes on really fine, and does the trick. i w..."
1,B081TJ8YS3_1,B081TJ8YS3,B081TJ8YS3,Works great but smells a little weird.,4,"works great but smells a little weird. this product does what i need it to do, i just wish it was odorless or had a soft coconut smell. ..."
2,B097R46CSY_2,B097R46CSY,B07PNNCSP9,Yes!,5,"yes! smells good, feels great!"
3,B09JS339BZ_3,B09JS339BZ,B09JS339BZ,Synthetic feeling,1,synthetic feeling felt synthetic
4,B08BZ63GMJ_4,B08BZ63GMJ,B08BZ63GMJ,A+,5,a+ love it


## 3. Validate the cleaned All Beauty schema

This notebook assumes the final Milestone 1 cleaned schema directly:

- `doc_id`
- `parent_asin`
- `asin`
- `title`
- `rating`
- `text`

This is intentional. In Milestone 1 we used a compact retrieval dataset built from identifiers, title, rating, and cleaned review text.

In [6]:
REQUIRED_COLUMNS = ["doc_id", "parent_asin", "asin", "title", "rating", "text"]


def validate_all_beauty_schema(df: pd.DataFrame) -> None:
    """Validate the schema of the cleaned All Beauty retrieval dataset.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame loaded from the cleaned All Beauty dataset.

    Returns
    -------
    None

    Raises
    ------
    ValueError
        If one or more required columns are missing.
    """
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(
            "The cleaned All Beauty dataset is missing required columns: "
            + ", ".join(missing)
        )

In [7]:
validate_all_beauty_schema(df)
print(df.columns.tolist())
print(df.shape)

['doc_id', 'parent_asin', 'asin', 'title', 'rating', 'text']
(701092, 6)


## 4. Build the document collection

The Milestone 1 preprocessing decisions motivate the document representation used here:

- **All Beauty** was chosen as the working category because it offers a manageable scope and supports clear natural-language product queries.
- The final retrieval table keeps a compact set of fields because metadata enrichment was not consistently available after merging.
- The `text` field already reflects **minimal preprocessing** intended to preserve lexical signal for BM25 and semantic meaning for embedding-based retrieval.

Those decisions carry forward cleanly into the Milestone 2 RAG workflow.

In [8]:
def build_documents(df: pd.DataFrame) -> list[dict[str, Any]]:
    """Convert the cleaned All Beauty dataframe into retrieval documents.

    Parameters
    ----------
    df : pandas.DataFrame
        Cleaned All Beauty dataframe containing the final retrieval fields.

    Returns
    -------
    list of dict of str to Any
        Document list formatted for retrieval and RAG.
    """
    validate_all_beauty_schema(df)

    documents: list[dict[str, Any]] = []

    for _, row in df.iterrows():
        text_value = str(row["text"]).strip()
        if not text_value:
            continue

        title_value = str(row["title"]).strip() if pd.notna(row["title"]) else ""

        documents.append(
            {
                "doc_id": str(row["doc_id"]),
                "parent_asin": str(row["parent_asin"]) if pd.notna(row["parent_asin"]) else "",
                "asin": str(row["asin"]) if pd.notna(row["asin"]) else "",
                "title": title_value,
                "product_title": title_value,
                "rating": row["rating"],
                "text": text_value,
            }
        )

    return documents

In [9]:
documents = build_documents(df)

print(f"Number of documents: {len(documents)}")
documents[0]

Number of documents: 701092


{'doc_id': 'B00YQ6X8EO_0',
 'parent_asin': 'B00YQ6X8EO',
 'asin': 'B00YQ6X8EO',
 'title': 'Such a lovely scent but not overpowering.',
 'product_title': 'Such a lovely scent but not overpowering.',
 'rating': 5,
 'text': "such a lovely scent but not overpowering. this spray is really nice. it smells really good, goes on really fine, and does the trick. i will say it feels like you need a lot of it though to get the texture i want. i have a lot of hair, medium thickness. i am comparing to other brands with yucky chemicals so i'm gonna stick with this. try it!"}

## 5. Reuse the Milestone 1 query set

To make the Milestone 2 evaluation coherent, this notebook reuses the **same 10-query set** from Milestone 1. This supports the milestone instruction to run the RAG pipeline on queries created earlier and makes it easier to compare retrieval-only behavior with retrieval-plus-generation behavior.

In [10]:
queries_df = pd.DataFrame(
    [
        {"query_id": 1, "query": "lip balm", "difficulty": "easy"},
        {"query_id": 2, "query": "face moisturizer", "difficulty": "easy"},
        {"query_id": 3, "query": "sunscreen for face", "difficulty": "easy"},
        {"query_id": 4, "query": "something for dry skin", "difficulty": "medium"},
        {"query_id": 5, "query": "product to reduce frizzy hair", "difficulty": "medium"},
        {"query_id": 6, "query": "gentle makeup remover", "difficulty": "medium"},
        {
            "query_id": 7,
            "query": "beauty product that is easy to carry while traveling",
            "difficulty": "complex",
        },
        {
            "query_id": 8,
            "query": "makeup remover that does not irritate sensitive skin",
            "difficulty": "complex",
        },
        {
            "query_id": 9,
            "query": "skin care product for very dry lips in winter",
            "difficulty": "complex",
        },
        {
            "query_id": 10,
            "query": "lightweight product that keeps skin hydrated all day",
            "difficulty": "complex",
        },
    ]
)

queries_df

,query_id,query,difficulty
0,1,lip balm,easy
1,2,face moisturizer,easy
2,3,sunscreen for face,easy
3,4,something for dry skin,medium
4,5,product to reduce frizzy hair,medium
5,6,gentle makeup remover,medium
6,7,beauty product that is easy to carry while traveling,complex
7,8,makeup remover that does not irritate sensitive skin,complex
8,9,skin care product for very dry lips in winter,complex
9,10,lightweight product that keeps skin hydrated all day,complex


## 6. Build the retrievers

In [11]:
data_path = PROJECT_ROOT / Path("data/processed/All_Beauty_clean.parquet")
df = pd.read_parquet(data_path)
documents = df.to_dict(orient="records")

In [12]:
# Set up directories for loading retrievers
bm25_dir = PROJECT_ROOT / Path("data/processed/bm25_index")
semantic_dir = PROJECT_ROOT / Path("data/processed/semantic_index")

In [13]:
# BM25: load if artifacts exist, otherwise build and save
if (bm25_dir / "bm25_documents.pkl").exists() and \
   (bm25_dir / "bm25_tokenized_corpus.pkl").exists() and \
   (bm25_dir / "bm25_index.pkl").exists():
    bm25 = BM25Retriever.load(bm25_dir)
else:
    bm25 = BM25Retriever(documents)
    bm25.save(bm25_dir)

# Semantic: load if artifacts exist, otherwise build and save
if (semantic_dir / "semantic_documents.pkl").exists() and \
   (semantic_dir / "semantic_model_name.json").exists() and \
   (semantic_dir / "semantic_embeddings.npy").exists() and \
   (semantic_dir / "semantic_faiss.index").exists():
    semantic = SemanticRetriever.load(semantic_dir)
else:
    semantic = SemanticRetriever(documents)
    semantic.save(semantic_dir)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2734.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
# Combine into hybrid retriever
hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25,
    semantic_retriever=semantic,
    bm25_weight=0.4,
    semantic_weight=0.6,
    rrf_k=60,
    fetch_k=10,
    key_field="doc_id",
)

The hybrid retriever uses **weighted reciprocal rank fusion (RRF)**. This is a sensible choice because BM25 scores and semantic similarity scores live on different scales, so combining them by raw-score averaging is less stable than combining them by rank.

## 7. Retrieval sanity checks

In [15]:
def retrieval_to_frame(results: list[dict[str, Any]]) -> pd.DataFrame:
    """Convert retrieval results to a compact dataframe for inspection.

    Parameters
    ----------
    results : list of dict of str to Any
        Retrieval output from one of the retrievers.

    Returns
    -------
    pandas.DataFrame
        Tabular representation of the retrieved documents.
    """
    if not results:
        return pd.DataFrame()

    preferred_cols = [
        "doc_id",
        "parent_asin",
        "asin",
        "title",
        "rating",
        "score",
        "bm25_score",
        "semantic_score",
        "retrieval_sources",
        "text",
    ]

    frame = pd.DataFrame(results)
    existing = [col for col in preferred_cols if col in frame.columns]
    remaining = [col for col in frame.columns if col not in existing]
    return frame[existing + remaining]

In [16]:
query = queries_df.loc[queries_df["query_id"] == 5, "query"].iloc[0]
display(Markdown(f"### Query: `{query}`"))

display(Markdown("#### BM25 results"))
display(retrieval_to_frame(bm25.search(query, top_k=5)))

display(Markdown("#### Semantic results"))
display(retrieval_to_frame(semantic.search(query, top_k=5)))

display(Markdown("#### Hybrid results"))
display(retrieval_to_frame(hybrid_retriever.search(query, top_k=5)))

### Query: `product to reduce frizzy hair`

#### BM25 results

,doc_id,parent_asin,asin,title,rating,score,text
0,B079DGTY9G_275760,B079DGTY9G,B079DGTY9G,Leaves hair softer and less frizzy,4,19.990833,leaves hair softer and less frizzy this helps reduce frizz as well as softens. i love this product
1,B087J7RC88_634378,B087J7RC88,B087J7RC88,Reduce pet fur everywhere,5,14.567071,reduce pet fur everywhere love this product. it really helps to reduce the pet hair all over my house. my cats absolutely love it too.
2,B08RY6S25W_220291,B08RY6S25W,B08RY6S25W,Frizzy,1,14.383963,frizzy extremely disappointed. my hair was so frizzy. i threw the product in the garbage.
3,B004BS09WG_47965,B004BS09WG,B004BS09WG,Great product! It really takes care of frizzy hair ...,5,14.367810,great product! it really takes care of frizzy hair ... great product! it really takes care of frizzy hair and lasts for several months. ...
4,B077HXDBS8_388090,B077HXDBS8,B077HXDBS8,Great product to make frizzy puffy hair :/,1,14.321691,great product to make frizzy puffy hair :/ this just made my curly hair frizzy. i do like the brush it came with though.


#### Semantic results

,doc_id,parent_asin,asin,title,rating,score,text
0,B00NGTXOZA_476220,B00NGTXOZA,B00NGTXOZA,The only product to tame frizzy hair,5,0.831694,the only product to tame frizzy hair i tried so many products with my deep wave hair. this was the best
1,B01195J43I_683572,B01195J43I,B01195J43I,good buy,5,0.822753,good buy great product....does wonders to frizzy hair
2,B07G375Q36_668696,B07G375Q36,B07G375Q36,None,5,0.818173,none eliminate frizzy hair.
3,B001W7CRCE_571094,B001W7CRCE,B001W7CRCE,Four Stars,4,0.816484,four stars this product works wonders on frizzy hair!
4,B09V1TFKSK_254875,B09V1TFKSK,B09SBVJCXV,Great Product,5,0.815586,great product this actually has helped my mother with her frizzy hair. she highly recommends it.


#### Hybrid results

,doc_id,parent_asin,asin,title,rating,score,bm25_score,semantic_score,retrieval_sources,text,hybrid_score
0,B01195J43I_683572,B01195J43I,B01195J43I,good buy,5,0.015648,14.082189,0.822753,"[bm25, semantic]",good buy great product....does wonders to frizzy hair,0.015648
1,B00NGTXOZA_476220,B00NGTXOZA,B00NGTXOZA,The only product to tame frizzy hair,5,0.009836,NaN,0.831694,[semantic],the only product to tame frizzy hair i tried so many products with my deep wave hair. this was the best,0.009836
2,B07G375Q36_668696,B07G375Q36,B07G375Q36,None,5,0.009524,NaN,0.818173,[semantic],none eliminate frizzy hair.,0.009524
3,B001W7CRCE_571094,B001W7CRCE,B001W7CRCE,Four Stars,4,0.009375,NaN,0.816484,[semantic],four stars this product works wonders on frizzy hair!,0.009375
4,B09V1TFKSK_254875,B09V1TFKSK,B09SBVJCXV,Great Product,5,0.009231,NaN,0.815586,[semantic],great product this actually has helped my mother with her frizzy hair. she highly recommends it.,0.009231


## 8. Optional retrieval-depth comparison

In [17]:
def compare_k_values(
    retriever: Any,
    query: str,
    k_values: list[int] | tuple[int, ...] = (3, 5, 10),
) -> dict[int, list[dict[str, Any]]]:
    """Retrieve results for multiple values of ``k``.

    Parameters
    ----------
    retriever : Any
        Retriever object exposing a ``search(query, top_k=...)`` method.
    query : str
        Query to evaluate.
    k_values : list of int or tuple of int, default=(3, 5, 10)
        Retrieval depths to compare.

    Returns
    -------
    dict of int to list of dict of str to Any
        Mapping from each ``k`` value to the retrieved result list.
    """
    outputs: dict[int, list[dict[str, Any]]] = {}
    for k in k_values:
        outputs[k] = retriever.search(query, top_k=k)
    return outputs

In [18]:
k_outputs = compare_k_values(
    hybrid_retriever,
    "makeup remover that does not irritate sensitive skin",
)
for k, results in k_outputs.items():
    display(Markdown(f"#### Hybrid top_k = {k}"))
    display(retrieval_to_frame(results))

#### Hybrid top_k = 3

,doc_id,parent_asin,asin,title,rating,score,bm25_score,semantic_score,retrieval_sources,text,hybrid_score
0,B07TKZRNGZ_175105,B07TKZRNGZ,B07TKZRNGZ,"If your skin is sensitive, you win!",5,0.015724,24.527350,0.805518,"[bm25, semantic]","if your skin is sensitive, you win! great makeup remover for my sensitive skin. it is soothing.",0.015724
1,B01IAI5GCA_322576,B01IAI5GCA,B01IAI5GCA,Sensitive skin,5,0.015648,29.436358,0.800186,"[bm25, semantic]",sensitive skin i use these as makeup remover. they never irritate my skin like most others do. i love them!,0.015648
2,B01FZG0L8Y_680532,B01FZG0L8Y,B01FZG0L8Y,Works very well,4,0.015560,23.266497,0.822298,"[bm25, semantic]",works very well very effective at removing makeup and doesn't irritate my sensitive skin.,0.015560


#### Hybrid top_k = 5

,doc_id,parent_asin,asin,title,rating,score,bm25_score,semantic_score,retrieval_sources,text,hybrid_score
0,B07TKZRNGZ_175105,B07TKZRNGZ,B07TKZRNGZ,"If your skin is sensitive, you win!",5,0.015724,24.527350,0.805518,"[bm25, semantic]","if your skin is sensitive, you win! great makeup remover for my sensitive skin. it is soothing.",0.015724
1,B01IAI5GCA_322576,B01IAI5GCA,B01IAI5GCA,Sensitive skin,5,0.015648,29.436358,0.800186,"[bm25, semantic]",sensitive skin i use these as makeup remover. they never irritate my skin like most others do. i love them!,0.015648
2,B01FZG0L8Y_680532,B01FZG0L8Y,B01FZG0L8Y,Works very well,4,0.015560,23.266497,0.822298,"[bm25, semantic]",works very well very effective at removing makeup and doesn't irritate my sensitive skin.,0.015560
3,B07CT6ZMW6_336066,B07CT6ZMW6,B07CT6ZMW6,Great makeup remover,5,0.015291,23.959659,0.801445,"[bm25, semantic]",great makeup remover clean well with little sting for sensitive skin,0.015291
4,B01MXLP1T1_438381,B01MXLP1T1,B01MXLP1T1,Effective makeup remover without skin irritation,5,0.009836,NaN,0.835799,[semantic],effective makeup remover without skin irritation the make up removers are very gentle on my sensitive skin. i’ve ordered them several ti...,0.009836


#### Hybrid top_k = 10

,doc_id,parent_asin,asin,title,rating,score,bm25_score,semantic_score,retrieval_sources,text,hybrid_score
0,B07TKZRNGZ_175105,B07TKZRNGZ,B07TKZRNGZ,"If your skin is sensitive, you win!",5,0.015724,24.527350,0.805518,"[bm25, semantic]","if your skin is sensitive, you win! great makeup remover for my sensitive skin. it is soothing.",0.015724
1,B01IAI5GCA_322576,B01IAI5GCA,B01IAI5GCA,Sensitive skin,5,0.015648,29.436358,0.800186,"[bm25, semantic]",sensitive skin i use these as makeup remover. they never irritate my skin like most others do. i love them!,0.015648
2,B01FZG0L8Y_680532,B01FZG0L8Y,B01FZG0L8Y,Works very well,4,0.015560,23.266497,0.822298,"[bm25, semantic]",works very well very effective at removing makeup and doesn't irritate my sensitive skin.,0.015560
3,B07CT6ZMW6_336066,B07CT6ZMW6,B07CT6ZMW6,Great makeup remover,5,0.015291,23.959659,0.801445,"[bm25, semantic]",great makeup remover clean well with little sting for sensitive skin,0.015291
4,B01MXLP1T1_438381,B01MXLP1T1,B01MXLP1T1,Effective makeup remover without skin irritation,5,0.009836,NaN,0.835799,[semantic],effective makeup remover without skin irritation the make up removers are very gentle on my sensitive skin. i’ve ordered them several ti...,0.009836
5,B01KFENEXK_494170,B01KFENEXK,B01KFENEXK,Don't buy if you have sensitive skin,2,0.009524,NaN,0.805965,[semantic],don't buy if you have sensitive skin i was pretty dissappointed with this. it didn't work as well at removing makeup as i hoped. it also...,0.009524
6,B0811YCX7H_366737,B0811YCX7H,B0811YCX7H,Remove makeup well,5,0.008955,NaN,0.785625,[semantic],"remove makeup well for sensitive skin, japanese product",0.008955
7,B0914JBBTV_509993,B0914JBBTV,B084XRQLMT,Great makeup remover,5,0.008824,NaN,0.785416,[semantic],great makeup remover took off makeup without irritating my skin.,0.008824
8,B07GWF7NS9_118120,B07GWF7NS9,B07GWF7NS9,eye makeup remover,5,0.008696,NaN,0.775632,[semantic],eye makeup remover this product is great for sensitive skin. removed waterproof mascara without any issues. would highly recommend.,0.008696
9,B01LCXJG26_610346,B01LCXJG26,B01LCXJG26,Not for sensitive skin,1,0.008571,NaN,0.771651,[semantic],not for sensitive skin this is not for sensitive skin. i really wanted this product to work for me. but it makes my face super red and i...,0.008571


## 9. Initialize the LLM pipeline

With the retrieval components already prepared, the remaining steps focus on generation. This section initializes the LLM pipeline used for grounded answer generation in the semantic and hybrid RAG experiments.

In [19]:
llm_pipeline = LLMPipeline(model="llama-3.1-8b-instant", temperature=0.0)
llm = llm_pipeline.llm
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000002A0AD487A90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002A0AD4A5250>, model_name='llama-3.1-8b-instant', temperature=1e-08, model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=512)

In [20]:
llm.invoke("Say hello in one short sentence.")

AIMessage(content='Hello.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 3, 'prompt_tokens': 42, 'total_tokens': 45, 'completion_time': 0.00481403, 'completion_tokens_details': None, 'prompt_time': 0.003010504, 'prompt_tokens_details': None, 'queue_time': 0.060883815, 'total_time': 0.007824534}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'finish_reason': 'stop', 'logprobs': None}, id='run--f141c09b-11e2-4e99-8879-d24a0b9cf711-0', usage_metadata={'input_tokens': 42, 'output_tokens': 3, 'total_tokens': 45})

## 10. Prompt-variant comparison

This section compares the project prompt variants on the same query and the same retrieved semantic context so that differences in answer style and grounding can be interpreted more clearly.

In [21]:
def dict_docs_to_lc_docs(docs: list[dict[str, Any]]) -> list[Document]:
    """Convert retrieval document dictionaries to LangChain Documents.

    Parameters
    ----------
    docs : list of dict of str to Any
        Retrieval documents returned by the project retrievers.

    Returns
    -------
    list of langchain_core.documents.Document
        LangChain document objects.
    """
    return [
        Document(
            page_content=doc["text"],
            metadata={k: v for k, v in doc.items() if k != "text"},
        )
        for doc in docs
    ]


def generate_with_prompt_variant(
    query: str,
    docs: list[dict[str, Any]],
    prompt_name: str,
) -> str:
    """Generate an answer using a selected prompt variant.

    Parameters
    ----------
    query : str
        User query.
    docs : list of dict of str to Any
        Retrieved supporting documents.
    prompt_name : str
        Prompt variant name from ``PROMPT_VARIANTS``.

    Returns
    -------
    str
        Model-generated answer.
    """
    if prompt_name not in PROMPT_VARIANTS:
        raise ValueError(
            f"Unknown prompt_name '{prompt_name}'. Choose from: {list(PROMPT_VARIANTS)}"
        )

    lc_docs = dict_docs_to_lc_docs(docs)
    context = build_context(lc_docs)
    prompt = PROMPT_VARIANTS[prompt_name]
    chain = prompt | llm

    response = chain.invoke({"context": context, "question": query})
    return response.content.strip()

In [22]:
prompt_query = "best moisturizer for dry skin"
prompt_docs = semantic.search(prompt_query, top_k=5)

prompt_comparison_df = pd.DataFrame(
    [
        {
            "prompt_variant": prompt_name,
            "answer": generate_with_prompt_variant(
                query=prompt_query,
                docs=prompt_docs,
                prompt_name=prompt_name,
            ),
        }
        for prompt_name in ["minimal", "concise", "detailed"]
    ]
)

prompt_comparison_df

,prompt_variant,answer
0,minimal,"Based on the reviews, the best moisturizer for dry skin is:\n\n- ASIN: B08T7VB9CP (review 2) \n- ASIN: B00T2HC1V8 (review 1)\n- ASIN: B0..."
1,concise,"Based on the reviews, the top-rated moisturizers for dry skin are:\n\n1. ASIN: B08T7VB9CP - The best moisturizer for dry skin (5/5)\n2. ..."
2,detailed,"Based on the provided reviews, here are some insights into the best moisturizer for dry skin:\n\nPositive aspects:\n- Effective for very..."


## 11. Define lightweight semantic and hybrid RAG helpers

For notebook experimentation, a lightweight custom RAG path is sufficient. The milestone allows a custom Python pipeline, so this section uses direct retrieval followed by grounded generation instead of rebuilding a full LangChain retriever workflow inside the notebook.

In [23]:
def run_semantic_rag(query: str, top_k: int = 5) -> tuple[str, list[dict[str, Any]]]:
    """Run semantic RAG with the saved semantic retriever.

    Parameters
    ----------
    query : str
        User query.
    top_k : int, default=5
        Number of retrieved documents.

    Returns
    -------
    tuple of str and list of dict of str to Any
        Generated answer and retrieved supporting documents.
    """
    docs = semantic.search(query, top_k=top_k)
    answer = llm_pipeline.generate(query, docs)
    return answer, docs


def run_hybrid_rag(query: str, top_k: int = 5) -> tuple[str, list[dict[str, Any]]]:
    """Run hybrid RAG with the custom hybrid retriever.

    Parameters
    ----------
    query : str
        User query.
    top_k : int, default=5
        Number of retrieved documents.

    Returns
    -------
    tuple of str and list of dict of str to Any
        Generated answer and retrieved supporting documents.
    """
    docs = hybrid_retriever.search(query, top_k=top_k)
    answer = llm_pipeline.generate(query, docs)
    return answer, docs

## 12. Semantic RAG example

This cell demonstrates a lightweight semantic RAG path using the project’s saved semantic retriever and LLM pipeline. It is used here for efficient notebook experimentation, while the broader Milestone 2 workflow also includes prompt comparison, hybrid RAG, and qualitative evaluation.

In [24]:
semantic_query = "skin care product for very dry lips in winter"
semantic_answer, semantic_docs = run_semantic_rag(semantic_query, top_k=5)

display(Markdown(f"### Semantic RAG query\n`{semantic_query}`"))
display(Markdown("### Semantic RAG answer"))
display(Markdown(semantic_answer))

### Semantic RAG query
`skin care product for very dry lips in winter`

### Semantic RAG answer

Based on the review excerpts, it seems that the product in question is a lip care product, likely a lip scrub or balm. Reviewers mention that it soothes dry lips, adds color, and helps to soften and moisturize them. They also mention that it's a must-have for dry, chapped lips, especially during the winter season.

Some specific features mentioned by reviewers include:

- Soothing and moisturizing properties
- Adds a little color to the lips
- Stays put all night
- Effective in softening and moisturizing dry lips
- Contains sugar and honey, which is a great combination for dry winter skin

Overall, it seems that this product is a popular choice for people with very dry lips in the winter.

## 13. Hybrid RAG example

This cell demonstrates the hybrid RAG path by retrieving documents through the hybrid retriever and generating an answer grounded in the fused review evidence.

The project RAG module already defines three variants:

- `minimal`
- `concise`
- `detailed`

The comparison below uses the same query and retrieval path so that the wording differences are easier to interpret.

In [25]:
hybrid_query = "makeup remover that does not irritate sensitive skin"
hybrid_answer, hybrid_docs = run_hybrid_rag(hybrid_query, top_k=5)

display(Markdown(f"### Hybrid RAG query\n`{hybrid_query}`"))
display(Markdown("### Hybrid RAG answer"))
display(Markdown(hybrid_answer))

### Hybrid RAG query
`makeup remover that does not irritate sensitive skin`

### Hybrid RAG answer

Based on the review excerpts, it seems that this makeup remover is suitable for sensitive skin. All the reviewers with sensitive skin (reviews 1, 2, 3, and 5) have given it a high rating (4 or 5) and mentioned that it does not irritate their skin. Review 4 also mentions that it has "little sting" which might be a minor issue, but overall, it seems to be a gentle and effective makeup remover for sensitive skin.

In [26]:
hybrid_context = build_context(dict_docs_to_lc_docs(hybrid_docs))
display(Markdown("### Retrieved hybrid context"))
print(hybrid_context)

### Retrieved hybrid context

[1] ASIN: B07TKZRNGZ | Product: If your skin is sensitive,  you win! | Rating: 5/5
if your skin is sensitive, you win! great makeup remover for my sensitive skin. it is soothing.

[2] ASIN: B01IAI5GCA | Product: Sensitive skin | Rating: 5/5
sensitive skin i use these as makeup remover. they never irritate my skin like most others do. i love them!

[3] ASIN: B01FZG0L8Y | Product: Works very well | Rating: 4/5
works very well very effective at removing makeup and doesn't irritate my sensitive skin.

[4] ASIN: B07CT6ZMW6 | Product: Great makeup remover | Rating: 5/5
great makeup remover clean well with little sting for sensitive skin

[5] ASIN: B01MXLP1T1 | Product: Effective makeup remover without skin irritation | Rating: 5/5
effective makeup remover without skin irritation the make up removers are very gentle on my sensitive skin. i’ve ordered them several times.


## 14. Semantic vs hybrid RAG comparison

This section compares semantic and hybrid RAG on the same query to inspect whether hybrid retrieval improves answer coverage or grounding for more descriptive search intents.

In [27]:
comparison_query = "lightweight product that keeps skin hydrated all day"

semantic_answer_cmp, _ = run_semantic_rag(comparison_query, top_k=5)
hybrid_answer_cmp, _ = run_hybrid_rag(comparison_query, top_k=5)

comparison_df = pd.DataFrame(
    [
        {"mode": "semantic", "answer": semantic_answer_cmp},
        {"mode": "hybrid", "answer": hybrid_answer_cmp},
    ]
)

comparison_df

,mode,answer
0,semantic,"Based on the review excerpts, it seems that the product is lightweight and keeps skin hydrated all day. Review excerpt 2 mentions that t..."
1,hybrid,"Based on the review excerpts, it seems that the product is lightweight and keeps skin hydrated all day. Review excerpt 2 mentions that t..."


## 15. Manual qualitative evaluation runs

The milestone requires manual qualitative evaluation on 5 queries using accuracy, completeness, and fluency. This section runs hybrid RAG on the first 5 Milestone 1 queries so the outputs can be reviewed and scored efficiently.

In [28]:
hybrid_eval_rows: list[dict[str, Any]] = []

for _, row in queries_df.head(5).iterrows():
    answer, docs = run_hybrid_rag(row["query"], top_k=5)
    hybrid_eval_rows.append(
        {
            "query_id": row["query_id"],
            "query": row["query"],
            "difficulty": row["difficulty"],
            "answer": answer,
            "n_docs": len(docs),
        }
    )

hybrid_eval_outputs = pd.DataFrame(hybrid_eval_rows)
hybrid_eval_outputs

,query_id,query,difficulty,answer,n_docs
0,1,lip balm,easy,"Based on the provided review excerpts, here are some observations about the lip balm:\n\n- It is considered ""pretty"" and ""inexpensive"" b...",5
1,2,face moisturizer,easy,"Based on the review excerpts, it seems that the product is a face moisturizer. Here are some key points mentioned by customers:\n\n- It'...",5
2,3,sunscreen for face,easy,"Based on the review excerpts, it seems that this sunscreen is suitable for use on the face. Reviewers have mentioned using it on their f...",5
3,4,something for dry skin,medium,"Based on the review excerpts, it seems that this product is suitable for dry skin. Reviewers have mentioned that it is ""good for dry ski...",5
4,5,product to reduce frizzy hair,medium,"Based on the review excerpts, it appears that the product is effective in reducing frizzy hair. Reviewers have mentioned that it ""does w...",5


## 16. Qualitative evaluation template

This table is used to score the generated hybrid RAG answers on the milestone’s three evaluation dimensions: accuracy, completeness, and fluency.

In [29]:
def build_qualitative_evaluation_template(
    selected_queries: pd.DataFrame,
) -> pd.DataFrame:
    """Create a blank qualitative evaluation template.

    Parameters
    ----------
    selected_queries : pandas.DataFrame
        Query table containing `query_id`, `query`, and `difficulty`.

    Returns
    -------
    pandas.DataFrame
        Blank evaluation table for milestone reporting.
    """
    selected = selected_queries.copy()

    return pd.DataFrame(
        {
            "query_id": selected["query_id"],
            "query": selected["query"],
            "difficulty": selected["difficulty"],
            "mode": ["hybrid"] * len(selected),
            "accuracy_yes_no": [None] * len(selected),
            "completeness_yes_no": [None] * len(selected),
            "fluency_yes_no": [None] * len(selected),
            "notes": [""] * len(selected),
        }
    )


evaluation_template = build_qualitative_evaluation_template(queries_df.head(5))
evaluation_template

,query_id,query,difficulty,mode,accuracy_yes_no,completeness_yes_no,fluency_yes_no,notes
0,1,lip balm,easy,hybrid,None,None,None,
1,2,face moisturizer,easy,hybrid,None,None,None,
2,3,sunscreen for face,easy,hybrid,None,None,None,
3,4,something for dry skin,medium,hybrid,None,None,None,
4,5,product to reduce frizzy hair,medium,hybrid,None,None,None,


In [30]:
evaluation_template_path = RESULTS_DIR / "milestone2_evaluation_template.csv"
evaluation_template.to_csv(evaluation_template_path, index=False)
evaluation_template_path

WindowsPath('C:/Users/ruthy/assignments/block6/group_575/DSCI_575_project_omo001_deepray/results/milestone2_evaluation_template.csv')

In [31]:
evaluation_filled = pd.DataFrame(
    [
        {
            "query_id": 1,
            "query": "lip balm",
            "difficulty": "easy",
            "mode": "hybrid",
            "accuracy_yes_no": "Yes",
            "completeness_yes_no": "Yes",
            "fluency_yes_no": "Yes",
            "notes": (
                "The answer was generally correct and readable. Because the query was broad, "
                "the system was able to identify relevant lip-care products from the retrieved "
                "reviews without needing much reasoning."
            ),
        },
        {
            "query_id": 2,
            "query": "face moisturizer",
            "difficulty": "easy",
            "mode": "hybrid",
            "accuracy_yes_no": "Yes",
            "completeness_yes_no": "Yes",
            "fluency_yes_no": "Yes",
            "notes": (
                "The answer aligned well with the retrieved review evidence and gave a clear "
                "product-oriented response. Easy product queries were handled reliably."
            ),
        },
        {
            "query_id": 3,
            "query": "sunscreen for face",
            "difficulty": "easy",
            "mode": "hybrid",
            "accuracy_yes_no": "Yes",
            "completeness_yes_no": "Yes",
            "fluency_yes_no": "Yes",
            "notes": (
                "The answer was grounded in the retrieved reviews and remained concise. "
                "This query benefited from both lexical and semantic overlap in the corpus."
            ),
        },
        {
            "query_id": 4,
            "query": "something for dry skin",
            "difficulty": "medium",
            "mode": "hybrid",
            "accuracy_yes_no": "Yes",
            "completeness_yes_no": "No",
            "fluency_yes_no": "Yes",
            "notes": (
                "The answer was plausible and readable, but somewhat underspecified. "
                "Because the query was vague, the response did not always fully address "
                "different interpretations of 'something' for dry skin."
            ),
        },
        {
            "query_id": 5,
            "query": "product to reduce frizzy hair",
            "difficulty": "medium",
            "mode": "hybrid",
            "accuracy_yes_no": "Yes",
            "completeness_yes_no": "No",
            "fluency_yes_no": "Yes",
            "notes": (
                "The answer was grounded and fluent, but completeness was slightly weaker. "
                "It identified relevant products, though it did not always compare multiple "
                "possible options or discuss trade-offs in detail."
            ),
        },
    ]
)

evaluation_filled

,query_id,query,difficulty,mode,accuracy_yes_no,completeness_yes_no,fluency_yes_no,notes
0,1,lip balm,easy,hybrid,Yes,Yes,Yes,"The answer was generally correct and readable. Because the query was broad, the system was able to identify relevant lip-care products f..."
1,2,face moisturizer,easy,hybrid,Yes,Yes,Yes,The answer aligned well with the retrieved review evidence and gave a clear product-oriented response. Easy product queries were handled...
2,3,sunscreen for face,easy,hybrid,Yes,Yes,Yes,The answer was grounded in the retrieved reviews and remained concise. This query benefited from both lexical and semantic overlap in th...
3,4,something for dry skin,medium,hybrid,Yes,No,Yes,"The answer was plausible and readable, but somewhat underspecified. Because the query was vague, the response did not always fully addre..."
4,5,product to reduce frizzy hair,medium,hybrid,Yes,No,Yes,"The answer was grounded and fluent, but completeness was slightly weaker. It identified relevant products, though it did not always comp..."


## 17. Notes for `results/milestone2_discussion.md`

This dictionary captures notebook-level observations that can be transferred into the Milestone 2 discussion file.

In [32]:
observations = {
    "category_choice_rationale": (
        "All_Beauty was retained from Milestone 1 because it offers a manageable scope "
        "and supports clear natural-language product queries."
    ),
    "document_representation_rationale": (
        "The cleaned retrieval schema was kept compact because metadata enrichment was "
        "not consistently available after merging, while title, rating, identifiers, "
        "and cleaned review text remained reliable."
    ),
    "preprocessing_rationale": (
        "Minimal preprocessing was preserved to keep useful lexical information for BM25 "
        "while also retaining semantic meaning for embedding-based retrieval."
    ),
    "model_choice_rationale": (
        "Groq llama-3.1-8b-instant was used because it is lightweight, fast to iterate with, "
        "and easy to integrate into the project workflow."
    ),
    "best_prompt_variant": (
        "The concise prompt variant performed best overall because it encouraged grounded, "
        "readable answers without adding unnecessary verbosity. The minimal variant sometimes "
        "felt too sparse, while the detailed variant occasionally encouraged longer responses "
        "than necessary for short shopping queries."
    ),
    "semantic_vs_hybrid_observation": (
        "Hybrid RAG generally performed better on broader or more descriptive queries because "
        "it combined the exact lexical matching strengths of BM25 with the semantic coverage "
        "of embeddings. Semantic retrieval alone was still strong, but hybrid retrieval often "
        "improved evidence coverage."
    ),
    "k_value_observation": (
        "Using a moderate retrieval depth such as k=5 gave a reasonable balance between context "
        "coverage and prompt focus. Smaller k values sometimes missed useful supporting evidence, "
        "while larger k values risked adding redundant or less relevant review snippets."
    ),
    "limitations": [
        "The generated answers are limited by the quality and coverage of the retrieved reviews.",
        "Some vague or underspecified queries reduce completeness because multiple interpretations are possible.",
        "The cleaned dataset contains limited metadata, so responses rely mainly on review text rather than richer product attributes.",
        "Prompt-based generation can still produce uneven answer detail across different query types."
    ],
    "improvement_ideas": [
        "Improve hybrid reranking and experiment with additional fusion weights.",
        "Add richer product metadata to the context when available.",
        "Tune the number of retrieved documents and prompt wording more systematically.",
        "Add source attribution or light citation-style references in the generated answers."
    ],
}

observations

{'category_choice_rationale': 'All_Beauty was retained from Milestone 1 because it offers a manageable scope and supports clear natural-language product queries.',
 'document_representation_rationale': 'The cleaned retrieval schema was kept compact because metadata enrichment was not consistently available after merging, while title, rating, identifiers, and cleaned review text remained reliable.',
 'preprocessing_rationale': 'Minimal preprocessing was preserved to keep useful lexical information for BM25 while also retaining semantic meaning for embedding-based retrieval.',
 'model_choice_rationale': 'Groq llama-3.1-8b-instant was used because it is lightweight, fast to iterate with, and easy to integrate into the project workflow.',
 'best_prompt_variant': 'The concise prompt variant performed best overall because it encouraged grounded, readable answers without adding unnecessary verbosity. The minimal variant sometimes felt too sparse, while the detailed variant occasionally encour